In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Recomendación minorista multimodal: uso de Gemini para recomendar artículos basados en imágenes y razonamiento de imágenes

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/retail/multimodal_retail_recommendations.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Ejecutar en Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fuse-cases%2Fretail%2Fmultimodal_retail_recommendations.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Ejecutar en Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/use-cases/retail/multimodal_retail_recommendations.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Abrir en Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/retail/multimodal_retail_recommendations.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> Ver en GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://goo.gle/4fX5Ige">
      <img width="32px" src="https://cdn.qwiklabs.com/assets/gcp_cloud-e3a77215f0b8bfa9b3f611c0d2208c7e8708ed31.svg" alt="Google Cloud logo"><br> Abrir en Skills
    </a>
  </td>
</table>

<div style="clear: both;"></div>


## Descripción general

Para las empresas minoristas (retail), los sistemas de recomendación mejoran la experiencia del cliente y, por lo tanto, pueden aumentar las ventas.

Este notebook muestra cómo puedes usar las capacidades multimodales del modelo Gemini 2.5 Flash para crear rápidamente un sistema de recomendación multimodal listo para usar.

## Escenario

El cliente te muestra su sala de estar:

| Foto del cliente |
|:-----:|
| <img src="https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/rooms/spacejoy-c0JoR_-2x3E-unsplash.jpg" width="80%"> |

A continuación se presentan cuatro opciones de sillas entre las que el cliente está tratando de decidir:

|Silla 1| Silla 2 | Silla 3 | Silla 4 |
|:-----:|:----:|:-----:|:----:|
| <img src="https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/cesar-couto-OB2F6CsMva8-unsplash.jpg" width="80%">|<img src="https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/daniil-silantev-1P6AnKDw6S8-unsplash.jpg" width="80%">|<img src="https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/ruslan-bardash-4kTbAMRAHtQ-unsplash.jpg" width="80%">|<img src="https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/scopic-ltd-NLlWwR4d3qU-unsplash.jpg" width="80%">|

¿Cómo puedes usar Gemini, un modelo multimodal, para ayudar al cliente a elegir la mejor opción y también explicar por qué?

### Objetivos

Tu objetivo principal es aprender a crear un sistema de recomendación que pueda proporcionar tanto recomendaciones como explicaciones utilizando el modelo multimodal Gemini 2.5 Flash.

En este notebook, comenzarás con una escena (por ejemplo, una sala de estar) y usarás el modelo Gemini para realizar una comprensión visual. También investigarás cómo se puede utilizar el modelo Gemini para recomendar un artículo (por ejemplo, una silla) a partir de una lista de artículos de mobiliario como entrada.

Al pasar por este notebook, aprenderás:
- cómo usar el modelo Gemini para realizar una comprensión visual
- cómo tener en cuenta la multimodalidad al crear prompts para el modelo Gemini
- cómo se puede utilizar el modelo Gemini para crear aplicaciones de recomendación minorista listas para usar

## Primeros pasos

In [ ]:
%pip install --upgrade google-genai

In [ ]:
from google import genai
from google.genai.types import GenerateContentConfig, Part

PROJECT_ID = "tu-proyecto-id"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## Uso del modelo Gemini

Gemini es un modelo multimodal que permite agregar imágenes y videos en prompts de texto o chat para obtener una respuesta de texto.

In [ ]:
MODEL_ID = "gemini-2.5-flash"

### Comprensión visual con Gemini

Aquí le pedirás al modelo Gemini que describa una habitación en detalle a partir de su imagen. Para ello, tienes que **combinar texto e imagen en un solo prompt**.

In [ ]:
# urls para imágenes de habitaciones
room_image_url = "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/rooms/spacejoy-c0JoR_-2x3E-unsplash.jpg"
room_image = Part.from_uri(file_uri=room_image_url, mime_type="image/jpeg")

prompt = "Describe qué es visible en esta habitación y la atmósfera general:"
contents = [
    room_image,
    prompt,
]

responses = client.models.generate_content_stream(model=MODEL_ID, contents=contents)

print("\n-------Respuesta--------")
for response in responses:
    print(response.text, end="")

### Generación de recomendaciones abiertas basadas en el conocimiento integrado

Usando la misma imagen, puedes pedirle al modelo que recomiende **un mueble** que encaje en ella junto con la descripción de la habitación.

Ten en cuenta que el modelo puede elegir **cualquier mueble** para recomendar en este caso, y puede hacerlo a partir de su propio conocimiento integrado.

In [ ]:
prompt1 = "Recomienda un nuevo mueble para esta habitación:"
prompt2 = "y explica el motivo en detalle"
contents = [prompt1, room_image, prompt2]

responses = client.models.generate_content_stream(model=MODEL_ID, contents=contents)

print("\n-------Respuesta--------")
for response in responses:
    print(response.text, end="")

### Generación de recomendaciones basadas en imágenes proporcionadas

En lugar de dejar la recomendación abierta, también puedes proporcionar una lista de artículos para que el modelo elija. Aquí descargarás algunas imágenes de sillas y las establecerás como opciones para que el modelo Gemini las recomiende. Esto es particularmente útil para las empresas minoristas que desean proporcionar recomendaciones a los usuarios basadas en el tipo de habitación que tienen y los artículos disponibles que ofrece la tienda.

In [ ]:
# Descargar y mostrar sillas de muestra
furniture_image_urls = [
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/cesar-couto-OB2F6CsMva8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/daniil-silantev-1P6AnKDw6S8-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/ruslan-bardash-4kTbAMRAHtQ-unsplash.jpg",
    "https://storage.googleapis.com/github-repo/img/gemini/retail-recommendations/furnitures/scopic-ltd-NLlWwR4d3qU-unsplash.jpg",
]

furniture_images = [
    Part.from_uri(file_uri=url, mime_type="image/jpeg") for url in furniture_image_urls
]

# Para recomendar un artículo de una selección, deberás etiquetar el número del artículo dentro del prompt.
# De esa manera, le estarás proporcionando al modelo una forma de referenciar cada imagen al plantear una pregunta.
# Etiquetar las imágenes dentro de tu prompt también ayuda a reducir las alucinaciones y, en general, produce mejores resultados.
contents = [
    "Considera las siguientes sillas:",
    "silla 1:",
    furniture_images[0],
    "silla 2:",
    furniture_images[1],
    "silla 3:",
    furniture_images[2],
    "silla 4:",
    furniture_images[3],
    "habitación:",
    room_image,
    "Eres un diseñador de interiores. Para cada silla, explica si sería apropiada para el estilo de la habitación:",
]

responses = client.models.generate_content_stream(model=MODEL_ID, contents=contents)

print("\n-------Respuesta--------")
for response in responses:
    print(response.text, end="")

También puedes devolver las respuestas en formato JSON, para que sea más fácil integrar las recomendaciones en un sistema de recomendación:

In [ ]:
contents = [
    "Considera las siguientes sillas:",
    "silla 1:",
    furniture_images[0],
    "silla 2:",
    furniture_images[1],
    "silla 3:",
    furniture_images[2],
    "silla 4:",
    furniture_images[3],
    "habitación:",
    room_image,
    "Eres un diseñador de interiores. Devuelve en JSON, para cada silla, si encajaría en la habitación, con una explicación:",
]

responses = client.models.generate_content_stream(
    model=MODEL_ID,
    contents=contents,
    config=GenerateContentConfig(response_mime_type="application/json"),
)

print("\n-------Respuesta--------")
for response in responses:
    print(response.text, end="")

## Conclusión

Este notebook mostró qué tan fácil es construir un sistema de recomendación multimodal utilizando Gemini para muebles, pero también puedes usar un enfoque similar en:

- recomendar ropa según una ocasión o una imagen del lugar
- recomendar papel tapiz según la habitación y los ajustes

Es posible que también desees explorar cómo puedes construir un sistema RAG (generación aumentada por recuperación) donde recuperes imágenes relevantes del inventario de tu tienda para los usuarios, quienes luego pueden usar Gemini para ayudar a identificar la opción más ideal de las diversas opciones proporcionadas, y también explicar el razonamiento a los usuarios.